# 🏷️ Fine-Tuning DistilBERT for NER (Restaurant Search) - Beginner Tutorial


Welcome! In this tutorial, we will learn:

- Why we fine-tune models like DistilBERT
- How a model behaves **before** and **after** fine-tuning
- Real-world examples showing the improvement
- How to make predictions using your fine-tuned model

This is a **complete beginner-level** guide! 🚀


## 1. Install Required Libraries

In [ ]:

!pip install -U transformers


## 2. Understanding Pretraining vs Fine-Tuning


- 🤖 **Pretraining**: DistilBERT is trained on a large amount of general text. It learns language but **does NOT know** about specific tasks like NER (Named Entity Recognition for restaurants).
- 🎯 **Fine-Tuning**: We teach the pretrained model a **specific task** (like recognizing cuisines, locations, etc.) using labeled data.

> **Real-life analogy**: Pretraining is like graduating from school, and fine-tuning is like doing a special course (e.g., becoming a chef, a doctor, etc.)!


## 3. Load Raw DistilBERT Model (Before Fine-Tuning)

In [ ]:

from transformers import AutoTokenizer, AutoModelForTokenClassification

# Load raw model
base_model_name = "distilbert-base-uncased"
tokenizer_base = AutoTokenizer.from_pretrained(base_model_name)
model_base = AutoModelForTokenClassification.from_pretrained(base_model_name, num_labels=7)  # Assume 7 labels


## 4. Helper Function for Entity Prediction

In [ ]:

def predict_entities(sentence, tokenizer, model):
    inputs = tokenizer(sentence, return_tensors="pt", truncation=True, is_split_into_words=False)
    outputs = model(**inputs)
    logits = outputs.logits
    predictions = logits.argmax(dim=-1)
    
    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
    predicted_labels = [str(pred.item()) for pred in predictions[0]]
    
    entities = []
    for token, label in zip(tokens, predicted_labels):
        if label != "0" and not token.startswith("["):
            entities.append((token, label))
    return entities


## 5. Predictions Using Raw DistilBERT (Before Fine-Tuning)

In [ ]:

examples = [
    "Find me an Italian restaurant in New York with outdoor seating.",
    "Locate a sushi place near San Francisco airport.",
    "Suggest French cuisine near Central Park."
]

for idx, sentence in enumerate(examples, 1):
    print(f"\nExample {idx}: {sentence}")
    entities = predict_entities(sentence, tokenizer_base, model_base)
    if entities:
        for token, label in entities:
            print(f"  Token: {token}, Label ID: {label}")
    else:
        print("  No useful entities detected (random/noisy labels).")


## 6. Load Fine-Tuned Model (After Fine-Tuning)

In [ ]:

# Path where your fine-tuned model is saved
fine_tuned_model_path = "./ner_model"

tokenizer_finetuned = AutoTokenizer.from_pretrained(fine_tuned_model_path)
model_finetuned = AutoModelForTokenClassification.from_pretrained(fine_tuned_model_path)


## 7. Predictions Using Fine-Tuned DistilBERT

In [ ]:

def predict_entities_finetuned(sentence, tokenizer, model):
    inputs = tokenizer(sentence, return_tensors="pt", truncation=True, is_split_into_words=False)
    outputs = model(**inputs)
    logits = outputs.logits
    predictions = logits.argmax(dim=-1)
    
    id2label = {v: k for k, v in model.config.label2id.items()}
    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
    predicted_labels = [id2label[pred.item()] for pred in predictions[0]]
    
    entities = []
    for token, label in zip(tokens, predicted_labels):
        if label != "O" and not token.startswith("["):
            entities.append((token, label))
    return entities

for idx, sentence in enumerate(examples, 1):
    print(f"\nExample {idx}: {sentence}")
    entities = predict_entities_finetuned(sentence, tokenizer_finetuned, model_finetuned)
    if entities:
        for token, label in entities:
            print(f"  Token: {token}, Entity: {label}")
    else:
        print("  No entities detected.")


## 8. Summary: Before vs After Fine-Tuning


| Step | Before Fine-Tuning | After Fine-Tuning |
|:--|:--|:--|
| **Understanding sentences** | ❌ Random / Noisy predictions | ✅ Correct detection of cuisine, location, features |
| **Recognizing restaurant terms** | ❌ Doesn't recognize | ✅ Recognizes |
| **Usefulness** | ❌ Not usable directly | ✅ Ready for real applications! |

🎯 **Fine-tuning helps a general-purpose model specialize for your task!**
